# Frontiers in Neurorobotics — scope drift deep-dive

Further work on run **`20260721_122750`** (2020–2026, full network, τ=5).

**Questions**
1. When did scope drift start?
2. How is the journal drifting (community mix)?
3. Is OOS genuine?
4. Do journal **sections** (or Research Topics) explain drift/OOS?
5. Should we expand scope, rename, and/or add sections?

**Baseline:** 2020  
**Inputs:** `output/scope_dashboard.html`, `output/drift_dashboard.html`, `output/network_maps.html`  
**Section/RT join:** run `python further_work/probe_sections_rts.py` (writes CSVs + summary JSON under `further_work/`).

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

HERE = Path.cwd()
if HERE.name != "further_work":
    # Allow running from repo root or further_work/
    if (HERE / "further_work").is_dir():
        HERE = HERE / "further_work"
sys.path.insert(0, str(HERE))

from neuro_analysis import (
    JOURNAL,
    PRIMARY_LABELS,
    contested_oos_titles,
    drift_trend,
    get_journal,
    load_dashboards,
    onset_year,
    primary_share_by_year,
    run_meta,
)

FIG = HERE / "figures"
FIG.mkdir(exist_ok=True)

scope, drift, maps = load_dashboards()
j = get_journal(scope)
mj = get_journal(maps)
trend = drift_trend(drift)
meta = run_meta(scope)

print(JOURNAL)
print("Run:", meta.get("run_timestamp"), "|", meta.get("generated_utc"))
print(
    "Source:",
    meta.get("bq_source_dataset"),
    "| years",
    meta.get("start_year"),
    "–",
    meta.get("end_year"),
)
print(
    "Articles:",
    j["articles"],
    "| all-years OOS%:",
    j["out_of_scope_pct"],
    "| primary clusters:",
    j["n_primary_clusters"],
)

## 1. When did drift start?

JSD measures how different each year’s community mix is from the **2020 baseline** distribution.  
Onset ≈ first year with JSD ≥ 0.20 (medium-drift band used in the drift dashboard).

In [ ]:
onset = onset_year(trend, threshold=0.20)
df_trend = pd.DataFrame(
    {
        "year": trend.get("years", []),
        "jsd": trend.get("jsd", []),
        "new_comm_pct": trend.get("new_comm", []),
        "entropy_delta": trend.get("entropy_delta", []),
        "articles": trend.get("articles", []),
    }
)
display(df_trend)
print("Onset year (JSD ≥ 0.20):", onset)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(df_trend["year"], df_trend["jsd"], marker="o", color="#1a4f8c", lw=2)
ax.axhline(0.20, color="#d97706", ls="--", lw=1, label="Medium (0.20)")
ax.axhline(0.30, color="#c93030", ls="--", lw=1, label="High (0.30)")
ax.set_xlabel("Year")
ax.set_ylabel("JSD vs 2020")
ax.set_title("Neurorobotics composition drift")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(FIG / "neuro_jsd.png", dpi=140, bbox_inches="tight")
plt.show()

## 2. How is it drifting?

Primary **set** is stable (same four communities). The story is **reweighting**: computer vision rises; neuroscience shrinks.

In [ ]:
ps = j.get("primary_shift") or {}
print("Primary set changed 2020 →", ps.get("latest_year"), ":", ps.get("changed"))
print("Gained:", ps.get("gained_labels"))
print("Lost:", ps.get("lost_labels"))
print("\n2020 top:")
display(pd.DataFrame(ps.get("baseline_top") or []))
print("Latest top:")
display(pd.DataFrame(ps.get("latest_top") or []))

shares = primary_share_by_year(mj)
df_shares = pd.DataFrame(shares)
display(df_shares)

fig, ax = plt.subplots(figsize=(8, 3.8))
palette = {
    "Computer vision": "#2c5fa3",
    "Therapeutic Movement Sciences": "#1f8a4c",
    "Autonomous Systems and Control": "#856DF0",
    "Neuroscience": "#d4a300",
}
for lab, color in palette.items():
    ax.plot(df_shares["year"], df_shares[lab], marker="o", label=lab, color=color, lw=2)
ax.set_ylabel("Share of journal (%)")
ax.set_xlabel("Year")
ax.set_title("Primary community mix")
ax.set_ylim(0, 50)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(FIG / "neuro_shares.png", dpi=140, bbox_inches="tight")
plt.show()

print("OOS % by year (stable band, not an explosion):")
display(pd.DataFrame(j.get("oos_by_year") or []))

## 3. Top communities & borderline/OOS decisions

All non-primary candidates were judged **out_of_scope** by the LLM borderline step (no amber rescues for this journal).

In [ ]:
display(pd.DataFrame(j.get("top_communities") or []))

decisions = (j.get("scope_borderline") or {}).get("decisions") or []
df_dec = pd.DataFrame(decisions)
if not df_dec.empty:
    display(df_dec[["comm_id", "label", "verdict", "reason"]])
print("Borderline cluster ids:", j.get("borderline_cluster_ids"))

## 4. Is OOS genuine?

- **Genuine far-field OOS:** materials/thermal/geology/cancer/etc. without robot–neural link → reject/redirect.
- **False / soft OOS:** neurorobotics titles (dexterous hand, BCI–robot, prosthetics, soft robotics) sitting in OOS Leiden clusters → expand/keep candidates.
- **Brand stretch:** social language robots, pure medical imaging CV → editorial choice (section vs desk-reject).

In [ ]:
contested = contested_oos_titles(mj, limit=50)
df_c = pd.DataFrame(contested)
print(f"Contested OOS titles (keyword overlap): {len(df_c)}")
display(df_c)

print("\nExample papers from dashboard sample:")
ex = pd.DataFrame(j.get("example_papers") or [])
if not ex.empty:
    cols = [
        c
        for c in ["year", "is_in_scope", "community_label", "title"]
        if c in ex.columns
    ]
    display(ex[cols])

## 4. Sections vs Research Topics — what drove drift?

Join path (canonical):

```sql
article a
  LEFT JOIN taxonomy t ON a.taxonomy_id = t.taxonomy_id
  LEFT JOIN research_topic rt ON a.article_research_topic_id = rt.research_topic_id
WHERE a.space_id = 1 AND a.is_deleted = FALSE
```

`t.type` is `Field Journal` / `Specialty Journal` / `Specialty Section`; `t.section` + publish/create dates give section name and launch timing when present.

**Sections:** Every Neurorobotics paper is `Specialty Journal` with **null** `section` (1,116 published 2020–26; 1,037/1,037 in the run). So we **cannot** attribute drift/OOS to a section launch.

**Research Topics:** ~80% of run papers have `article_research_topic_id`. RT create cohorts peak **2020–2022** (with JSD onset). Compare OOS% RT vs non-RT and by RT create year.

Re-run: `python further_work/probe_sections_rts.py`

In [ ]:
from neuro_analysis import load_rt_tables, load_section_rt_summary

summary = load_section_rt_summary()
launch, by_topic, by_year_rt = load_rt_tables()

print(summary["verdict"])
print(
    f"Matched {summary['n_matched_rdm']}/{summary['n_scatter']} · "
    f"sections={summary['n_with_section']} · "
    f"types={summary.get('taxonomy_types')} · "
    f"in RT={summary['n_with_rt']} · "
    f"OOS RT {summary['oos_pct_rt']}% vs non-RT {summary['oos_pct_non_rt']}%"
)

display(launch.rename(columns={"rt_launch_year": "RT create year"}))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(
    launch["rt_launch_year"],
    launch["n_articles"],
    color="#1a4f8c",
    alpha=0.85,
    label="Articles",
)
ax2 = ax.twinx()
ax2.plot(
    launch["rt_launch_year"],
    launch["oos_pct"],
    color="#c93030",
    marker="o",
    lw=2,
    label="OOS %",
)
ax.set_xlabel("RT create year (launch cohort)")
ax.set_ylabel("Articles in run")
ax2.set_ylabel("OOS %")
ax.set_title("Neurorobotics — RT launch cohorts vs OOS")
fig.tight_layout()
fig.savefig(FIG / "rt_launch_vs_oos.png", dpi=140, bbox_inches="tight")
plt.show()

print("Highest-OOS Research Topics (by OOS paper count):")
cols = [
    c
    for c in ["rt_launch_year", "n", "oos_n", "oos_pct", "research_topic_title"]
    if c in by_topic.columns
]
display(by_topic[cols].head(15))

print("OOS by article year × in Research Topic:")
display(by_year_rt)

## 5. Recommendations

| Question | Recommendation |
|---|---|
| **Did a section launch cause drift?** | **No** — journal has no sections today. Drift is mix shift + RT-heavy intake, not section architecture. |
| **Did RT launches contribute?** | Partly: RT papers are slightly more OOS (≈22% vs ≈16% non-RT). Peak RT create cohorts **2020–2022** align with JSD onset; some ML/vision RTs have high OOS shares. |
| **Expand scope?** | Yes, selectively: embodied/robot vision, therapeutic robotics, neural↔robot control. Not materials/IoT/traffic/generic NLP. |
| **Rename?** | Strong case — name undersells CV/control and over-promises neuroscience. e.g. *Neurorobotics and Embodied AI*. |
| **Add sections?** | Yes as *forward* architecture (not historical cause): (1) Neural interfaces & motor neuroscience (2) Therapeutic & assistive robotics (3) Embodied vision (robot-gated) (4) Learning & control for physical agents (5) optional social HRI |
| **OOS genuine?** | Far-field yes; audit clusters 0/3/20 for false OOS before treating rates as pure quality signal. |

**Strategy choice**
- **A — Broaden + rename** toward embodied AI (match demand).
- **B — Tighten** to neural–robot + rehab; desk-reject pure CV.

## 6. Build PDF brief

Writes `further_work/Neurorobotics_Scope_Drift_Brief.pdf`.

In [ ]:
import subprocess

pdf_script = HERE / "build_neurorobotics_brief_pdf.py"
result = subprocess.run(
    [sys.executable, str(pdf_script)],
    cwd=str(HERE.parent),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
print("returncode", result.returncode)
assert result.returncode == 0, "PDF build failed"
print("PDF:", HERE / "Neurorobotics_Scope_Drift_Brief.pdf")